# Leaflet cluster map of talk locations

Assuming you are working in a Linux or Windows Subsystem for Linux environment, you may need to install some dependencies. Assuming a clean installation, the following will be needed:

```bash
sudo apt install jupyter
sudo apt install python3-pip
pip install python-frontmatter getorg --upgrade
```

After which you can run this from the `_talks/` directory, via:

```bash
 jupyter nbconvert --to notebook --execute talkmap.ipynb --output talkmap_out.ipynb
```
 
The `_talks/` directory contains `.md` files of all your talks. This scrapes the location YAML field from each `.md` file, geolocates it with `geopy/Nominatim`, and uses the `getorg` library to output data, HTML, and Javascript for a standalone cluster map.

In [1]:
# Start by installing the dependencies
!pip install python-frontmatter getorg --upgrade
import frontmatter
import glob
import getorg
from geopy import Nominatim
from geopy.exc import GeocoderTimedOut


[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: pip install --upgrade pip


Iywidgets and ipyleaflet support disabled. You must be in a Jupyter notebook to use this feature.
Error raised:
No module named 'ipyleaflet'
Check that you have enabled ipyleaflet in Jupyter with:
    jupyter nbextension enable --py ipyleaflet


In [2]:
# Collect the Markdown files
g = glob.glob("_talks/*.md")

In [3]:
# Set the default timeout, in seconds
TIMEOUT = 5

# Prepare to geolocate
geocoder = Nominatim(user_agent="academicpages.github.io")
location_dict = {}
location = ""
permalink = ""
title = ""

In the event that this times out with an error, double check to make sure that the location is can be properly geolocated.

In [4]:
# Perform geolocation
for file in g:
    # Read the file
    data = frontmatter.load(file)
    data = data.to_dict()

    # Press on if the location is not present
    if 'location' not in data:
        continue

    # Prepare the description
    title = data['title'].strip()
    venue = data['venue'].strip()
    location = data['location'].strip()
    description = f"{title}<br />{venue}; {location}"

    # Geocode the location and report the status
    try:
        location_dict[description] = geocoder.geocode(location, timeout=TIMEOUT)
        print(description, location_dict[description])
    except ValueError as ex:
        print(f"Error: geocode failed on input {location} with message {ex}")
    except GeocoderTimedOut as ex:
        print(f"Error: geocode timed out on input {location} with message {ex}")
    except Exception as ex:
        print(f"An unhandled exception occurred while processing input {location} with message {ex}")

Bergamot NMT<br />European Language Grid Conference; Brussels, Belgium Bruxelles - Brussel, Brussel-Hoofdstad - Bruxelles-Capitale, Région de Bruxelles-Capitale - Brussels Hoofdstedelijk Gewest, België / Belgique / Belgien


Common Voice<br />Celtic Language Technology Workshop; Dublin, Ireland Dublin, County Dublin, Leinster, Éire / Ireland


Deep Speech + Yandex STT<br />Berlin NLP Meetup; Berlin, Germany Berlin, Deutschland
Persistent Homology<br />Big Data Week Berlin; Berlin, Germany Berlin, Deutschland


Deep Speech<br />RE•WORK Deep Learning Summit; Boston, USA Boston, Suffolk County, Massachusetts, United States


Architecture and Use of GAT<br />GGF Seattle; Seattle, WA, USA Seattle, King County, Washington, United States


Architecture and Use of GAT<br />GGF Summer School; Vico Equense (Naples), Italy Vico Equense, Napoli, Campania, Italia


Common Voice + Deep Speech<br />International Symposium on Linguistic Patterns in Spontaneous Speech; Taipei, Taiwan 臺北市, 臺灣
Architecture and Use of GAT<br />GGF Summer School; Vico Equense (Naples), Italy Vico Equense, Napoli, Campania, Italia
Deep Speech<br />RE•WORK Deep Learning Summit; Boston, USA Boston, Suffolk County, Massachusetts, United States
Deep Speech<br />Taipei Machine Learning Meetup; Taipei, Taiwan 臺北市, 臺灣
Deep Speech<br />Berlin Machine Learning Meetup; Berlin, Germany Berlin, Deutschland


Gödel’s Poetry<br />ai4math Workshop 2026; Potsdam, Germany Potsdam, Brandenburg, Deutschland


Architecture and Use of GAT<br />National e-Science Centre; Edinburgh, Scotland City of Edinburgh, Alba / Scotland, United Kingdom


In [5]:
# Save the map
m = getorg.orgmap.create_map_obj()
getorg.orgmap.output_html_cluster_map(location_dict, folder_name="talkmap", hashed_usernames=False)

'Written map to talkmap/'